## Deep Research

Multiple agents researching, web searching, summarizing, and creating a report based on an equipment failure query. 

In [2]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
#from messenger import send_email, push

In [3]:
load_dotenv(override=True)

True

In [4]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5



## We will build 3 Agents:

1. The Search Agent: searches the web for information
2. The Analyzer Agent: analyzes the tool log
3. The Writer Agent: writes a robust report


And then 3 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Analyzer Agent

In [14]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
find the root cause of the failure of the tools/equipment.
You report the root cause in a succinct, 1 paragraph, 100 word summary. Reply only with the summary.
"""
task = "JR electrostatic chuck electrical and mechanical failure"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [15]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [16]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

The most likely root cause of JR electrostatic chuck electrical/mechanical failure is degradation of the chuck’s conductive path and ceramic surface under plasma/thermal cycling. In J-R ESCs, exposed electrical connectors, center-tap/balancing connections, or conductive gas-conduit paths can be eroded, cracked, or severed during use or removal, leading to loss of electrical continuity even when the joint still looks mechanically intact. Separately, nonuniform charge separation and surface erosion/roughness increase chucking/dechucking issues, backside gas loss, wafer sliding, and premature failure. In short: progressive plasma erosion plus mechanical stress compromises both electrical contact and surface integrity, causing malfunction.

## Agent 2: The Analyzer Agent

### Using Structured Outputs, and including a description of the fields

In [17]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [18]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [19]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [20]:

result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find general causes and failure modes for electrostatic chucks, including electrical and mechanical issues, to ground the investigation.', query='electrostatic chuck electrical failure mechanical failure causes'), WebSearchItem(reason='Look for documentation or case studies specifically mentioning JR electrostatic chuck failures or JR brand/product lines.', query='JR electrostatic chuck failure electrical mechanical'), WebSearchItem(reason='Search technical papers and maintenance guides describing common symptoms such as arcing, dielectric breakdown, delamination, cracking, and clamp loss.', query='electrostatic chuck failure modes arcing dielectric breakdown delamination cracking'), WebSearchItem(reason='Find troubleshooting and diagnostic procedures for electrostatic chucks used in semiconductor equipment.', query='electrostatic chuck troubleshooting diagnostics maintenance guide'), WebSearchItem(reason='Search for manufacturer manuals or

## Agent 3: The Writer Agent

In [21]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 3 paragraphs and a total of 300 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## Agent 4: The email agent

In [22]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [23]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [24]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [25]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [26]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Demo of the Agents

In [27]:
query ="JR ESC electrical and mechanical failure"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Research complete!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Research complete!


In [28]:
report

ReportData(short_summary='The research points to two plausible interpretations of “JR ESC electrical and mechanical failure.” In Japanese railway contexts, JR East failures are usually caused by internal electrical or mechanical defects in rolling stock, power supply, or catenary equipment; however, “ESC” can also refer to a Johnsen-Rahbek electrostatic chuck, where failures are typically due to loss of clamping performance and material degradation. The safest conclusion is that the phrase is ambiguous and should be clarified before a definitive root-cause analysis is made.', markdown_report='## JR ESC Electrical and Mechanical Failure\n\nThe available research suggests that the phrase **“JR ESC electrical and mechanical failure” is ambiguous** and may refer to two different technical domains. In the railway context, JR East materials and JTSB investigations consistently show that service disruptions are most often caused by **internal equipment faults** rather than external events. Co